In [ ]:
!pip install -U bitsandbytes transformers accelerate peft trl datasets

In [ ]:
import transformers
import peft
import trl

print(f"Transformers: {transformers.__version__}")
print(f"PEFT: {peft.__version__}")
print(f"TRL: {trl.__version__}")

In [ ]:
import torch
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, EarlyStoppingCallback, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# 0. SETTINGS & STABILITY
set_seed(42)
MODEL_NAME = "meta-llama/Llama-3.2-1B"
DATASET_PATH = "/kaggle/input/myrbidata/rbi_sft_dataset_3000_corrected.json"
TOKEN = ""  # <-- MAKE SURE THIS IS SET(use your Hugging Face Access Token)
# 1. LOAD MODEL & TOKENIZER (Defining the 'model' variable)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=TOKEN
)
model = prepare_model_for_kbit_training(model)

# 2. APPLY LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 3. DATA PREP (Loading 'dataset')
raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")

def format_instruction(sample):
    return {"text": f"### Instruction:\n{sample['instruction']}\n\n### Response:\n{sample['response']}"}

dataset = raw_dataset.map(format_instruction, remove_columns=raw_dataset.column_names)
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# 4. SFT CONFIG (Using 'max_length' for TRL 0.12+)
sft_config = SFTConfig(
    output_dir="./rbi-expert-model",
    max_length=512,
    dataset_text_field="text",
    num_train_epochs=8,              # Mentor request
    eval_strategy="epoch",           # Monitor Val Loss
    save_strategy="epoch",
    load_best_model_at_end=True,     # Patience requirement
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    bf16=True,
    optim="paged_adamw_32bit",
    report_to="none",
    save_total_limit=2
)
# 5. INITIALIZE & TRAIN
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=sft_config,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🔥 Variables defined. Starting RBI training loop...")
trainer.train()

In [ ]:
import os

# 1. Find the latest checkpoint folder in your output directory
output_dir = "./rbi-expert-model" # or whatever you named your output_dir
checkpoints = [os.path.join(output_dir, d) for d in os.listdir(output_dir) if "checkpoint" in d]
latest_checkpoint = max(checkpoints, key=os.path.getmtime)

print(f"✅ Found your work! Loading from: {latest_checkpoint}")

# 2. Save the final version properly so you can stop the GPU
trainer.save_model("./rbi_expert_final_v1")
tokenizer.save_pretrained("./rbi_expert_final_v1")

print("🚀 Step 1 Complete: Your best model is now saved as 'rbi_expert_final_v1'")

In [ ]:
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# 1. Load the base model (The "Body")
# We use the original model name here
base_model_path = "meta-llama/Llama-3.2-1B"
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    token="" # Put your Hugging Face token here 
)
# 2. Load your "Expertise" (The "Brain")
model = PeftModel.from_pretrained(model, "./rbi_expert_final_v1")
# 3. Ask a question
tokenizer = AutoTokenizer.from_pretrained(base_model_path, token="")
# Updated Inference Code
prompt = "### Instruction:\nwhat is the maximum leverage ratio for nbfc-mfis?.\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# We increase max_new_tokens to 512 and add temperature for better flow
outputs = model.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    do_sample=True,
    repetition_penalty=1.2,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import time
import re
import json
import torch

# 1. Load the dataset
with open('/kaggle/input/myrbidata/rbi_sft_dataset_3000_corrected.json', 'r') as f:
    rbi_data = json.load(f)

# --- Helper Functions for Metrics ---
def clean_text(text):
    """Removes punctuation and converts to lowercase for fair comparison."""
    return re.sub(r'[^\w\s]', '', text.lower()).strip()

def find_best_match(user_query, dataset):
    """Finds the response in the dataset that shares the most words with the query."""
    user_words = set(clean_text(user_query).split())
    best_item = None
    max_overlap = 0
    for item in dataset:
        inst_words = set(clean_text(item['instruction']).split())
        overlap = len(user_words.intersection(inst_words))
        if overlap > max_overlap:
            max_overlap = overlap
            best_item = item

    # If we share at least 3 significant words, consider it a match
    if max_overlap >= 3:
        return best_item['response'], best_item['instruction']
    return "N/A (New Question)", None

def get_nlp_metrics(pred, truth):
    """Calculates word-based Precision, Recall, and F1."""
    if truth == "N/A (New Question)": return 0.0, 0.0, 0.0
    p_words = clean_text(pred).split()
    t_words = clean_text(truth).split()

    p_set, t_set = set(p_words), set(t_words)
    inter = p_set.intersection(t_set)
    prec = len(inter) / len(p_set) if p_set else 0
    rec = len(inter) / len(t_set) if t_set else 0
    f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) else 0
    return prec, rec, f1

# --- Main Interaction ---
user_prompt = input("Enter your RBI Question: ")

# 2. Find Ground Truth
ground_truth, matched_inst = find_best_match(user_prompt, rbi_data)

print(f"\n Thinking...")
if matched_inst:
    print(f" Matched with Dataset: '{matched_inst}'")

# 3. Inference
start_time = time.time()
formatted_prompt = f"### Instruction:\n{user_prompt}\n\n### Response:\n"
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
prompt_len = inputs['input_ids'].shape[1]
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
latency = time.time() - start_time

# 4. Processing Output
prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1].strip()
gen_len = outputs[0].shape[0] - prompt_len

# 5. Metrics Calculation
precision, recall, f1 = get_nlp_metrics(prediction, ground_truth)

ref_nums = set(re.findall(r'\d+', ground_truth))
pred_nums = set(re.findall(r'\d+', prediction))
is_accurate = ref_nums.issubset(pred_nums) if ref_nums else True
hallucinated = (len(pred_nums - ref_nums) > 0) if ref_nums else False
# 6. Cost Calculation (Llama 3.2 1B Market Rate)
cost = (prompt_len * 0.00000001) + (gen_len * 0.00000002)

# --- DISPLAY RESULTS ---
print("="*50)
print(f"MODEL RESPONSE:\n{prediction}")
print("="*50)
print(f" Accuracy:      {'100%' if is_accurate else 'Factual Error'}")
print(f" Hallucination: {'None' if not hallucinated else 'Detected'}")
print(f" F1-Score:      {f1:.4f}  (Precision: {precision:.2f}, Recall: {recall:.2f})")
print(f" Latency:       {latency:.2f} seconds")
print(f" Tokens Used:   {prompt_len + gen_len} ({prompt_len} in / {gen_len} out)")
print(f" Est. Cost:     ${cost:.8f}")
print("="*50)

In [ ]:
!pip install evaluate rouge_score bleu

In [ ]:
import time
import re
import json
import torch
from evaluate import load

# 1. Setup Evaluators & Data
rouge_evaluator = load("rouge")
bleu_evaluator = load("bleu")

with open('/kaggle/input/myrbidata/rbi_sft_dataset_3000_corrected.json', 'r') as f:
    rbi_data = json.load(f)

# --- Helper Functions ---
def clean_text(text):
    return re.sub(r'[^\w\s]', '', text.lower()).strip()
def find_best_match(user_query, dataset):
    user_words = set(clean_text(user_query).split())
    best_item = None
    max_overlap = 0
    for item in dataset:
        inst_words = set(clean_text(item['instruction']).split())
        overlap = len(user_words.intersection(inst_words))
        if overlap > max_overlap:
            max_overlap, best_item = overlap, item
    if max_overlap >= 3:
        return best_item['response'], best_item['instruction']
    return None, None

# --- Main Interaction Loop ---
print(" RBI Expert Model Evaluator (Type 'exit' to stop)")

while True:
    user_prompt = input("\nEnter RBI Question: ")
    if user_prompt.lower() == 'exit': break
# 1. Find Ground Truth
    ground_truth, matched_inst = find_best_match(user_prompt, rbi_data)

    # 2. Model Inference
    print(" Thinking...")
    start_time = time.time()
    formatted_prompt = f"### Instruction:\n{user_prompt}\n\n### Response:\n"
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")
    prompt_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    latency = time.time() - start_time
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).split("### Response:\n")[-1].strip()
    gen_len = outputs[0].shape[0] - prompt_len
    print("\n" + "="*60)
    print(f"MODEL RESPONSE:\n{prediction}")
    print("="*60)

    if ground_truth:
        # Traditional NLP Metrics (ROUGE & BLEU)
        rouge_res = rouge_evaluator.compute(predictions=[prediction], references=[ground_truth])
        bleu_res = bleu_evaluator.compute(predictions=[prediction], references=[[ground_truth]])

        # Word-Level Metrics (Precision, Recall, F1)
        p_set, t_set = set(clean_text(prediction).split()), set(clean_text(ground_truth).split())
        inter = p_set.intersection(t_set)
        prec = len(inter) / len(p_set) if p_set else 0
        rec = len(inter) / len(t_set) if t_set else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) else 0

        # Factual Accuracy & Hallucination
        ref_nums = set(re.findall(r'\d+', ground_truth))
        pred_nums = set(re.findall(r'\d+', prediction))
        is_accurate = ref_nums.issubset(pred_nums) if ref_nums else True
        hallucinated = (len(pred_nums - ref_nums) > 0) if ref_nums else False
        # Display Summary
        print(f" NLP METRICS:")
        print(f"   ROUGE-L: {rouge_res['rougeL']:.4f} | BLEU: {bleu_res['bleu']:.4f}")
        print(f"   F1-Score: {f1:.4f} (Prec: {prec:.2f}, Rec: {rec:.2f})")

        print(f"\n COMPLIANCE CHECK:")
        print(f"   Accuracy:      {'100%' if is_accurate else ' Factual Error'}")
        print(f"   Hallucination: {'None' if not hallucinated else ' Detected'}")

        print(f"\n PERFORMANCE & COST:")
        print(f"   Latency: {latency:.2f}s | Tokens: {prompt_len + gen_len}")
        print(f"   Est. Cost: ${((prompt_len * 0.01) + (gen_len * 0.02)) / 1_000_000:.8f}")
    else:
        print("Prompt not in dataset - Metrics skipped.")
    print("="*60)

In [ ]:
import shutil
import os

# Define the folder containing your final fine-tuned model
MODEL_FOLDER = "./rbi_expert_final_v1"

# Define the name for the output zip file
ZIP_FILENAME = "rbi_expert_final_v1"

# 1. Validate folder existence
print(f" Checking for model folder: {MODEL_FOLDER}")
if not os.path.exists(MODEL_FOLDER):
    raise FileNotFoundError(f"❌ Model folder not found at {MODEL_FOLDER}. Please ensure the model was saved correctly.")
print("Model folder found.")

# 2. Create the zip archive
print(f"Zipping model folder '{MODEL_FOLDER}' to '{ZIP_FILENAME}.zip'...")
shutil.make_archive(ZIP_FILENAME, 'zip', MODEL_FOLDER)

zip_file_path = f"{ZIP_FILENAME}.zip"

print(f"ZIP created: {zip_file_path}")
print(f"Size: {os.path.getsize(zip_file_path) / (1024*1024):.2f} MB")

# In Kaggle, files saved to the output directory are automatically available.
# No explicit download command is needed.
print("Model zipped and saved to output. In Kaggle, this will be available in your notebook's output.")
